# Projet — Exploitation du dateset CL-Drive

## Estimation de la charge cognitive du conducteur

Ce TP vient après les TD1, TD2, TD3 et TD4.

Les TD ont déjà permis de travailler :

- la compréhension du papier CL-Drive ;
- le protocole expérimental ;
- la segmentation en fenêtres de 10 s ;
- le prétraitement EEG ;
- l'extraction des features EEG.

Le point de départ du TP est donc le dossier généré à la fin du TD4 :

```text
EEG_Features_10s/
```

Ce TP ne revient pas sur le calcul des features. Il exploite les features déjà extraites pour construire un pipeline d'apprentissage automatique.

## Objectif du TP

Construire un pipeline complet :

```text
EEG_Features_10s
→ Normalized_Features_10s/EEG
→ Normalized_Features_10s_With_Label/EEG
→ Dataset EEG supervisé
→ Classification de la charge cognitive
→ Évaluation
→ Interprétation
```

Dans un premier temps, on se limite à l'EEG uniquement.

La multimodalité, c'est-à-dire l'ajout de ECG, EDA et Gaze, sera proposée uniquement comme extension à la fin du sujet.

## 1. Structure attendue des dossiers

Avant de commencer, le dossier de travail doit contenir au minimum :

```text
Data/
├── EEG/ID_x
│   ├── ... fichiers level_1, level_2, ..., level_9
│   ├── ... fichiers baseline
│   └── ... fichiers filtered_*
│
├── EEG_Features_10s/
│   ├── ID1_EEG_features.csv
│   ├── ID2_EEG_features.csv
│   └── ...
│
├── Labels/
│   ├── ID1.csv
│   ├── ID2.csv
│   └── ...
```

Le TP va générer deux nouveaux dossiers :

```text
Data/
├── Normalized_Features_10s/
│   └── EEG/
│       ├── norm_ID1_EEG_features.csv
│       ├── norm_ID2_EEG_features.csv
│       └── ...
│
├── Normalized_Features_10s_With_Label/(avec colonnes Level et Label)
│   └── EEG/
│       ├── norm_ID1_EEG_features.csv
│       ├── norm_ID2_EEG_features.csv
│       └── ...
```

## Question 

Pourquoi ne faut-il pas entraîner directement les modèles sur les fichiers `EEG_Features_10s`  ?

### Réponse 

...

In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

BASE_PATH = Path("Data")
if not BASE_PATH.exists():
    raise FileNotFoundError(f"Le dossier de données {BASE_PATH!s} est introuvable. Lancez le notebook depuis la racine du projet.")

EEG_FEATURE_DIR = BASE_PATH / "EEG_Features_10s"
LABEL_DIR = BASE_PATH / "Labels"
NORMALIZED_ROOT = BASE_PATH / "Normalized_Features_10s"
NORMALIZED_EEG_DIR = NORMALIZED_ROOT / "EEG"
LABELED_ROOT = BASE_PATH / "Normalized_Features_10s_With_Label"
LABELED_EEG_DIR = LABELED_ROOT / "EEG"

NORMALIZED_EEG_DIR.mkdir(parents=True, exist_ok=True)
LABELED_EEG_DIR.mkdir(parents=True, exist_ok=True)

METADATA_COLUMNS = ["Participant", "File", "Window", "Start_Time", "End_Time", "Channel"]

In [6]:
from scipy.signal import welch
from scipy.stats import entropy

FS = 256
WINDOW_SEC = 10
WINDOW_SAMPLES = FS * WINDOW_SEC
EEG_BANDS = {
    "delta": (1, 4),
    "theta": (4, 8),
    "alpha": (8, 12),
    "beta": (12, 30),
    "gamma": (30, 45),
}

def compute_psd_band_features(signal, fs=FS):
    if len(signal) < fs: 
        return {}
    freqs, psd = welch(signal, fs=fs, nperseg=min(512, len(signal)), scaling="density")
    features = {}
    for band_name, (fmin, fmax) in EEG_BANDS.items():
        idx = (freqs >= fmin) & (freqs < fmax)
        band_psd = psd[idx]
        if len(band_psd) == 0:
            features.update({f"{band_name}_{stat}": 0 for stat in ["pow", "mean", "max", "min", "median"]})
            continue
        features.update({
            f"{band_name}_pow": np.sum(band_psd),
            f"{band_name}_mean": np.mean(band_psd),
            f"{band_name}_max": np.max(band_psd),
            f"{band_name}_min": np.min(band_psd),
            f"{band_name}_median": np.median(band_psd),
        })
    return features

def compute_spectral_entropy_bands(signal, fs=FS):
    freqs, psd = welch(signal, fs=fs, nperseg=min(512, len(signal)))
    features = {}
    for band_name, (fmin, fmax) in EEG_BANDS.items():
        idx = (freqs >= fmin) & (freqs < fmax)
        band_psd = psd[idx]
        if len(band_psd) == 0 or np.sum(band_psd) == 0:
            features[f"{band_name}_entropy"] = 0
            continue
        p = band_psd / np.sum(band_psd)
        features[f"{band_name}_entropy"] = entropy(p, base=2)
    return features

def compute_hjorth(signal):
    signal = np.asarray(signal)
    if len(signal) < 2 or np.var(signal) == 0:
        return {"hjorth_mobility": 0, "hjorth_complexity": 0}
    diff1 = np.diff(signal)
    mobility = np.sqrt(np.var(diff1) / np.var(signal))
    diff2 = np.diff(diff1)
    complexity = np.sqrt(np.var(diff2) / np.var(diff1)) / mobility if mobility != 0 else 0
    return {"hjorth_mobility": mobility, "hjorth_complexity": complexity}

def lempel_ziv_complexity(signal):
    signal = np.asarray(signal)
    if len(signal) == 0:
        return 0
    median = np.median(signal)
    binary = (signal > median).astype(int)
    s = "".join(map(str, binary.tolist()))
    n = len(s)
    if n <= 1:
        return 0
    complexity = 1
    i = 1
    sub_str = s[0]
    while i < n:
        if s[:i+1] not in sub_str:
            complexity += 1
            sub_str += s[i]
        i += 1
    return complexity / (n / np.log2(n)) if n > 1 else 0

def higuchi_fd(signal, kmax=10):
    signal = np.asarray(signal)
    n = len(signal)
    if n < kmax * 2:
        return 0
    L = []
    for k in range(1, kmax + 1):
        Lk = []
        for m in range(k):
            Lmk = 0.0
            steps = int(np.floor((n - m) / k))
            if steps <= 1:
                continue
            for i in range(1, steps):
                Lmk += abs(signal[m + i*k] - signal[m + (i-1)*k])
            Lmk = Lmk * (n - 1) / (steps * k)
            Lk.append(Lmk)
        if Lk:
            L.append(np.mean(Lk))
    if len(L) < 2:
        return 0
    x = np.log(1.0 / np.arange(1, len(L) + 1))
    y = np.log(L)
    if np.any(np.isnan(y)) or np.any(np.isinf(y)):
        return 0
    return -np.polyfit(x, y, 1)[0]

def compute_raw_features(signal):
    signal = np.asarray(signal, dtype=float)
    signal = np.nan_to_num(signal, nan=0.0)
    return {
        "mean": np.mean(signal),
        "min": np.min(signal),
        "max": np.max(signal),
        "median": np.median(signal),
        "var": np.var(signal),
        "std": np.std(signal),
    }

def extract_eeg_features(signal, fs=FS):
    signal = np.asarray(signal, dtype=float)
    signal = np.nan_to_num(signal, nan=0.0)
    features = {}
    features.update(compute_psd_band_features(signal, fs))
    features.update(compute_spectral_entropy_bands(signal, fs))
    features.update(compute_hjorth(signal))
    features["lz_complexity"] = lempel_ziv_complexity(signal)
    features["higuchi_fd"] = higuchi_fd(signal)
    features.update(compute_raw_features(signal))
    return features

def process_eeg_file(file_path, participant_id):
    df = pd.read_csv(file_path)
    eeg_channels = [col for col in df.columns if col in ["AF7", "AF8", "TP9", "TP10"]]
    if not eeg_channels:
        eeg_channels = [col for col in df.columns if col not in ["Timestamp", "time", "index"]]
    records = []
    for start in range(0, len(df) - WINDOW_SAMPLES + 1, WINDOW_SAMPLES):
        window = df.iloc[start:start + WINDOW_SAMPLES]
        window_time = start / FS
        for ch in eeg_channels:
            signal = window[ch].values
            features = extract_eeg_features(signal, FS)
            record = {
                "Participant": participant_id,
                "File": file_path.name,
                "Window": start // WINDOW_SAMPLES,
                "Start_Time": window_time,
                "End_Time": window_time + WINDOW_SEC,
                "Channel": ch,
            }
            record.update(features)
            records.append(record)
    return pd.DataFrame(records)

def generate_eeg_features():
    out_dir = BASE_PATH / "EEG_Features_10s"
    out_dir.mkdir(parents=True, exist_ok=True)
    files_generated = 0
    for participant_folder in sorted((BASE_PATH / "EEG").iterdir()):
        if not participant_folder.is_dir():
            continue
        participant_id = participant_folder.name
        output_file = out_dir / f"{participant_id}_EEG_features.csv"
        dfs = []
        for data_file in sorted(participant_folder.glob("*.csv")):
            if data_file.name.lower().startswith("eeg_data_") or data_file.name.lower().startswith("eeg_baseline_"):
                df_feat = process_eeg_file(data_file, participant_id)
                if not df_feat.empty:
                    dfs.append(df_feat)
        if dfs:
            pd.concat(dfs, ignore_index=True).to_csv(output_file, index=False)
            print(f"Créé {output_file.relative_to(BASE_PATH.parent)} ({sum(len(df) for df in dfs)} lignes)")
            files_generated += 1
    return files_generated

nb_files = generate_eeg_features()
print(f"Génération terminée : {nb_files} fichiers créés dans EEG_Features_10s")


Créé Data\EEG_Features_10s\1030_EEG_features.csv (1068 lignes)
Créé Data\EEG_Features_10s\1105_EEG_features.csv (1068 lignes)
Créé Data\EEG_Features_10s\1106_EEG_features.csv (1080 lignes)


C:\Users\dell\AppData\Local\Temp\ipykernel_14556\776130178.py:100: RuntimeWarning: divide by zero encountered in log
  y = np.log(L)
C:\Users\dell\AppData\Local\Temp\ipykernel_14556\776130178.py:100: RuntimeWarning: divide by zero encountered in log
  y = np.log(L)
C:\Users\dell\AppData\Local\Temp\ipykernel_14556\776130178.py:100: RuntimeWarning: divide by zero encountered in log
  y = np.log(L)


Créé Data\EEG_Features_10s\1241_EEG_features.csv (1080 lignes)


C:\Users\dell\AppData\Local\Temp\ipykernel_14556\776130178.py:100: RuntimeWarning: divide by zero encountered in log
  y = np.log(L)


Créé Data\EEG_Features_10s\1271_EEG_features.csv (1004 lignes)


...

Créé Data\EEG_Features_10s\1868_EEG_features.csv (772 lignes)


C:\Users\dell\AppData\Local\Temp\ipykernel_14556\776130178.py:100: RuntimeWarning: divide by zero encountered in log
  y = np.log(L)


Créé Data\EEG_Features_10s\1892_EEG_features.csv (1060 lignes)


C:\Users\dell\AppData\Local\Temp\ipykernel_14556\776130178.py:100: RuntimeWarning: divide by zero encountered in log
  y = np.log(L)
C:\Users\dell\AppData\Local\Temp\ipykernel_14556\776130178.py:100: RuntimeWarning: divide by zero encountered in log
  y = np.log(L)
C:\Users\dell\AppData\Local\Temp\ipykernel_14556\776130178.py:100: RuntimeWarning: divide by zero encountered in log
  y = np.log(L)


Créé Data\EEG_Features_10s\1953_EEG_features.csv (1080 lignes)
Génération terminée : 21 fichiers créés dans EEG_Features_10s


## 2. Normalisation des features EEG

À la fin du TD4, chaque fichier CSV contient des features EEG calculées sur des fenêtres de 10 secondes.

La normalisation doit suivre deux étapes :

### Étape 1 — Normalisation par la baseline du sujet

Pour chaque sujet, les fichiers de baseline servent à calculer une valeur moyenne de référence pour chaque feature :

$$
\mu_{baseline}^{(s,f)} = \frac{1}{N}\sum_{i=1}^{N} x_i^{(s,f)}
$$

où :

- $s$ désigne le sujet ;
- $f$ désigne la feature ;
- $x_i^{(s,f)}$ désigne la valeur de la feature pendant la baseline.

Chaque valeur de feature dans les fichiers de tâche est ensuite divisée par la moyenne de baseline correspondante :

$$
x_{norm}^{(s,f)} = \frac{ x^{(s,f)} }{ \mu_{baseline}^{(s,f)} }
$$

### Étape 2 — Standardisation z-score

On applique ensuite une standardisation :

$$
z = \frac{x - \mu}{\sigma}
$$

Cela permet d’obtenir des features centrées et réduites. Cette étape devra toutefois être réalisée plus loin dans le pipeline, après la séparation des données entre les ensembles d’entraînement et de test (voir section 8 ci-dessous).

## Question

Quel est l'intérêt de la normalisation par baseline dans des signaux physiologiques ?

### Réponse 

...

In [2]:
def get_feature_columns(df, metadata_columns=METADATA_COLUMNS):
    """
    Retourne les colonnes numériques correspondant aux features.
    Les colonnes de métadonnées ne doivent pas être normalisées.
    """
    return [col for col in df.columns
            if col not in metadata_columns and pd.api.types.is_numeric_dtype(df[col])]


def compute_baseline_averages(feature_dir):
    """
    Calcule, pour chaque Participant, la moyenne de baseline de chaque feature.
    La baseline est identifiée par les fichiers contenant 'baseline' dans le nom.
    """
    baseline_avgs = {}
    for csv_file in sorted(feature_dir.glob("*.csv")):
        df = pd.read_csv(csv_file)
        if df.empty:
            continue
        if "Participant" in df.columns:
            participant_id = str(df["Participant"].iloc[0])
        else:
            participant_id = csv_file.stem.split("_")[0]
        
        # Filtrer les lignes de baseline en vérifiant le nom du fichier
        baseline_mask = df['File'].str.contains('baseline', case=False, na=False)
        baseline_df = df[baseline_mask]
        
        if baseline_df.empty:
            continue
        
        feature_cols = get_feature_columns(baseline_df)
        if not feature_cols:
            continue
        avg_values = baseline_df[feature_cols].mean()
        baseline_avgs.setdefault(participant_id, {}).update(avg_values.to_dict())
    return baseline_avgs


def normalize_by_baseline(df, participant_id, baseline_avgs):
    """
    Divise chaque feature par sa moyenne de baseline pour le sujet considéré.
    """
    df_norm = df.copy()
    feature_cols = get_feature_columns(df_norm)
    subject_baseline = baseline_avgs.get(str(participant_id), {})
    for col in feature_cols:
        baseline_value = subject_baseline.get(col, np.nan)
        if pd.isna(baseline_value) or baseline_value == 0:
            continue
        df_norm[col] = df_norm[col] / baseline_value
    return df_norm


def run_eeg_normalization():
    """
    Génère les fichiers du dossier :
    Normalized_Features_10s/EEG
    à partir du dossier :
    EEG_Features_10s
    
    Chaque fichier est d'abord filtré pour exclure les enregistrements de baseline,
    puis les features sont normalisées par rapport à la baseline du sujet.
    """
    baseline_avgs = compute_baseline_averages(EEG_FEATURE_DIR)
    for csv_file in sorted(EEG_FEATURE_DIR.glob("*.csv")):
        df = pd.read_csv(csv_file)
        if df.empty:
            continue
        participant_id = str(df["Participant"].iloc[0]) if "Participant" in df.columns else csv_file.stem.split("_")[0]
        
        # Extraire seulement les enregistrements de tâche (pas baseline)
        task_mask = ~df['File'].str.contains('baseline', case=False, na=False)
        df_task = df[task_mask].copy()
        
        if df_task.empty:
            print(f"Aucun enregistrement de tâche trouvé dans {csv_file.name}")
            continue
        
        if str(participant_id) not in baseline_avgs:
            print(f"Aucun profil baseline trouvé pour {participant_id} dans {csv_file.name}")
            continue
        
        df_norm = normalize_by_baseline(df_task, participant_id, baseline_avgs)
        output_file = NORMALIZED_EEG_DIR / f"norm_{csv_file.name}"
        df_norm.to_csv(output_file, index=False)
        print(f"Normalisé : {csv_file.name} → {output_file.name} ({len(df_norm)} lignes)")

In [3]:
run_eeg_normalization()

Normalisé : 1030_EEG_features.csv → norm_1030_EEG_features.csv (640 lignes)
Normalisé : 1105_EEG_features.csv → norm_1105_EEG_features.csv (636 lignes)
Normalisé : 1106_EEG_features.csv → norm_1106_EEG_features.csv (648 lignes)
Normalisé : 1241_EEG_features.csv → norm_1241_EEG_features.csv (648 lignes)
Normalisé : 1271_EEG_features.csv → norm_1271_EEG_features.csv (572 lignes)
Normalisé : 1314_EEG_features.csv → norm_1314_EEG_features.csv (560 lignes)
Normalisé : 1323_EEG_features.csv → norm_1323_EEG_features.csv (648 lignes)
Normalisé : 1337_EEG_features.csv → norm_1337_EEG_features.csv (648 lignes)
Normalisé : 1372_EEG_features.csv → norm_1372_EEG_features.csv (648 lignes)
Normalisé : 1417_EEG_features.csv → norm_1417_EEG_features.csv (648 lignes)
Normalisé : 1434_EEG_features.csv → norm_1434_EEG_features.csv (608 lignes)
Normalisé : 1544_EEG_features.csv → norm_1544_EEG_features.csv (604 lignes)
Normalisé : 1547_EEG_features.csv → norm_1547_EEG_features.csv (548 lignes)
Normalisé : 

## 3. Vérification du dossier `Normalized_Features_10s/EEG`

Après exécution de la normalisation, vérifiez que le dossier contient bien des fichiers `norm_*.csv`.

## Question

Pourquoi les fichiers de baseline ne sont-ils pas copiés dans le dossier normalisé final ?

### Réponse 

...

In [ ]:
# Vérification du dossier Normalized_Features_10s/EEG


## 4. Ajout des colonnes `Level` et `Label`

Les fichiers normalisés ne contiennent pas encore la cible d'apprentissage.

Il faut maintenant associer chaque fenêtre de 10 secondes à son score PAAS.

Les labels sont stockés dans le dossier :

```text
Labels/
```

Chaque fichier de labels correspond à un sujet, par exemple :

```text
Labels/ID1.csv
Labels/ID2.csv
...
```

Dans ces fichiers, on suppose une structure du type :

| time | lvl_1 | lvl_2 | ... | lvl_9 |
|---:|---:|---:|---|---:|
| 10 | 2 | 3 | ... | 5 |
| 20 | 2 | 4 | ... | 6 |
| ... | ... | ... | ... | ... |

Pour une fenêtre d'indice `Window`, le temps associé est :

$$
time = (Window + 1) \times 10
$$

Le niveau du scénario est extrait du nom du fichier avec une expression régulière :

```text
level_1 → Level = 1
level_2 → Level = 2
...
level_9 → Level = 9
```

Le score PAAS est ensuite récupéré dans la colonne :

```text
lvl_<Level>
```

Exemple : si `Level = 4`, on lit la colonne `lvl_4`.


In [4]:
def extract_level_from_filename(file_name):
    """
    Extrait le niveau de scénario à partir du nom de fichier.

    Exemple :
    filtered_level_3.csv → 3
    norm_filtered_level_8.csv → 8
    """
    name = Path(file_name).name if not isinstance(file_name, str) else file_name
    match = re.search(r"level[_-]?(\d+)", name, flags=re.IGNORECASE)
    return int(match.group(1)) if match else None


def get_label_for_row(row, labels_df):
    """
    Retourne le score PAAS correspondant à une ligne de features.
    """
    try:
        window = int(row["Window"])
    except (KeyError, ValueError, TypeError):
        return np.nan
    time_stamp = (window + 1) * 10
    level = row.get("Level") if "Level" in row else None
    if pd.isna(level):
        level = extract_level_from_filename(row.get("File", ""))
    if level is None:
        return np.nan
    label_col = f"lvl_{int(level)}"
    if label_col not in labels_df.columns:
        return np.nan
    match = labels_df[labels_df["time"] == time_stamp]
    if match.empty:
        return np.nan
    return match.iloc[0][label_col]


def attach_labels_eeg():
    """
    Génère les fichiers du dossier :
    Normalized_Features_10s_With_Label/EEG
    """
    for csv_file in sorted(NORMALIZED_EEG_DIR.glob("norm*.csv")):
        df = pd.read_csv(csv_file)
        if df.empty:
            continue
        participant_id = str(df["Participant"].iloc[0]) if "Participant" in df.columns else csv_file.stem.split("_")[0]
        label_file = LABEL_DIR / f"{participant_id}.csv"
        if not label_file.exists():
            print(f"Labels manquants pour {participant_id} → {label_file}")
            continue
        labels_df = pd.read_csv(label_file)
        if "time" not in labels_df.columns:
            print(f"Colonne time manquante dans {label_file}")
            continue
        labels_df = labels_df.copy()
        labels_df["time"] = pd.to_numeric(labels_df["time"], errors="coerce")
        labels_df = labels_df.dropna(subset=["time"])
        labels_df["time"] = labels_df["time"].astype(int)
        df["Level"] = df["File"].apply(extract_level_from_filename)
        df["Label"] = df.apply(lambda row: get_label_for_row(row, labels_df), axis=1)
        df = df.dropna(subset=["Label"]).reset_index(drop=True)
        output_file = LABELED_EEG_DIR / csv_file.name
        df.to_csv(output_file, index=False)
        print(f"Labellisé : {csv_file.name} → {output_file.name}")

In [5]:
attach_labels_eeg()

Labellisé : norm_1030_EEG_features.csv → norm_1030_EEG_features.csv
Labellisé : norm_1105_EEG_features.csv → norm_1105_EEG_features.csv
Labellisé : norm_1106_EEG_features.csv → norm_1106_EEG_features.csv
Labellisé : norm_1241_EEG_features.csv → norm_1241_EEG_features.csv
Labellisé : norm_1271_EEG_features.csv → norm_1271_EEG_features.csv
Labellisé : norm_1314_EEG_features.csv → norm_1314_EEG_features.csv
Labellisé : norm_1323_EEG_features.csv → norm_1323_EEG_features.csv
Labellisé : norm_1337_EEG_features.csv → norm_1337_EEG_features.csv
Labellisé : norm_1372_EEG_features.csv → norm_1372_EEG_features.csv
Labellisé : norm_1417_EEG_features.csv → norm_1417_EEG_features.csv
Labellisé : norm_1434_EEG_features.csv → norm_1434_EEG_features.csv
Labellisé : norm_1544_EEG_features.csv → norm_1544_EEG_features.csv
Labellisé : norm_1547_EEG_features.csv → norm_1547_EEG_features.csv
Labellisé : norm_1595_EEG_features.csv → norm_1595_EEG_features.csv
Labellisé : norm_1629_EEG_features.csv → norm_16

## 5. Vérification du dossier `Normalized_Features_10s_With_Label/EEG`

Le dossier final doit contenir des fichiers CSV avec au moins :

- les métadonnées : `Participant`, `File`, `Window`, `Channel`, `Start_Time` et `End_Time` ;
- les features EEG normalisées ;
- la colonne `Level` ;
- la colonne `Label`.

## Question

Quelle est la différence entre `Level` et `Label` dans ce TP ? Pourquoi faut-il ajouter à la fois `Level` et `Label` ?

### Réponse

...

In [ ]:
# Vérification du dossier Normalized_Features_10s_With_Label/EEG


## 6. Construction du dataset EEG supervisé

Une fois les fichiers normalisés et labellisés générés, on peut les concaténer pour construire un tableau unique.

Chaque ligne représente une fenêtre EEG de 10 secondes pour un canal.

On construit ensuite deux problèmes possibles :

### Classification binaire

| Score PAAS | Classe |
|---:|---|
| 1 à 4 | faible |
| 5 à 9 | élevée |

### Classification ternaire, extension

| Score PAAS | Classe |
|---:|---|
| 1 à 3 | faible |
| 4 à 6 | moyenne |
| 7 à 9 | élevée |

Dans ce TP, l'objectif principal est la classification binaire.

In [6]:
def load_labeled_eeg_dataset():
    """
    Concatène tous les fichiers CSV du dossier Normalized_Features_10s_With_Label/EEG.
    """
    df_list = []
    for csv_file in sorted(LABELED_EEG_DIR.glob("*.csv")):
        df = pd.read_csv(csv_file)
        if not df.empty:
            df_list.append(df)
    return pd.concat(df_list, ignore_index=True) if df_list else pd.DataFrame()


df = load_labeled_eeg_dataset()
print(df.shape)
df.head()

(12552, 48)


,Participant,File,Window,Start_Time,End_Time,Channel,delta_pow,delta_mean,delta_max,delta_min,...,lz_complexity,higuchi_fd,mean,min,max,median,var,std,Level,Label
0,1030,eeg_data_level_1.csv,0,0.0,10.0,TP9,0.856790,0.856790,0.978902,1.043254,...,1.0,0.925341,0.913883,1.404039,1.593062,1.297807,1.923925,1.624835,1,2.0
1,1030,eeg_data_level_1.csv,0,0.0,10.0,AF7,0.148661,0.148661,0.192358,0.096629,...,1.0,0.968926,0.949440,0.499440,0.435953,1.009406,0.180306,0.497417,1,2.0
2,1030,eeg_data_level_1.csv,0,0.0,10.0,AF8,0.304550,0.304550,0.218725,0.480203,...,1.0,0.983270,0.806580,0.550404,1.006359,0.865205,0.355542,0.698491,1,2.0
3,1030,eeg_data_level_1.csv,0,0.0,10.0,TP10,0.761109,0.761109,0.659238,1.116578,...,1.0,1.010364,0.923903,1.347979,0.981913,0.913272,0.842424,1.075178,1,2.0
4,1030,eeg_data_level_1.csv,1,10.0,20.0,TP9,1.572384,1.572384,1.147303,2.600321,...,1.0,0.818848,1.089273,1.720011,1.140812,0.985372,1.478190,1.424231,1,2.0


In [7]:
# Création des cibles de classification.
df["Label"] = pd.to_numeric(df["Label"], errors="coerce")
df = df.dropna(subset=["Label"]).reset_index(drop=True)
df["Label"] = df["Label"].astype(int)

df["Label_Binary"] = df["Label"].apply(lambda x: 0 if x <= 4 else 1)
# Extension ternaire éventuelle.
df["Label_Ternary"] = df["Label"].apply(lambda x: 0 if x <= 3 else (1 if x <= 6 else 2))

## 7. Préparation de la matrice d'apprentissage

On doit séparer :

- les métadonnées ;
- les features numériques EEG ;
- la cible d'apprentissage.

## Question

Pourquoi ne faut-il pas inclure `Participant`, `File`, `Window`, `Level` ou `Label` dans les features du modèle ?

### Réponse 

...

In [8]:
# Préparation des données d’entraînement
meta_cols = ["Participant", "File", "Window", "Start_Time", "End_Time", "Channel", "Level", "Label"]
feature_cols = [col for col in df.columns if col not in meta_cols + ["Label_Binary", "Label_Ternary"]]

X = df[feature_cols]
y = df["Label_Binary"]

print("Nombre d'exemples :", len(X))
print("Nombre de features :", len(feature_cols))
print("Exemples de features :", feature_cols[:10])

Nombre d'exemples : 12552
Nombre de features : 40
Exemples de features : ['delta_pow', 'delta_mean', 'delta_max', 'delta_min', 'delta_median', 'theta_pow', 'theta_mean', 'theta_max', 'theta_min', 'theta_median']


## 8. Classification EEG — premiers modèles

On teste plusieurs modèles classiques :

- LDA ;
- SVM ;
- Random Forest ;
- KNN ;
- Naive Bayes ;
- Decision Tree ;
- AdaBoost ;
- MLP.

La normalisation `StandardScaler` est placée dans le `sklearn.pipeline.Pipeline` pour éviter une fuite de données entre apprentissage et test. Il faut ajuster le `StandardScaler` uniquement sur les données d’entraînement :

`scaler.fit_transform(X_train)`

Puis appliquer la transformation aux données de test avec :

`scaler.transform(X_test)`

In [9]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import make_scorer, f1_score

models = {
    "LDA": LinearDiscriminantAnalysis(),
    "SVM": SVC(kernel="rbf", probability=True, class_weight="balanced", random_state=42),
    "RandomForest": RandomForestClassifier(n_estimators=100, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "NaiveBayes": GaussianNB(),
    "DecisionTree": DecisionTreeClassifier(random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42),
    "MLP": MLPClassifier(hidden_layer_sizes=(100,), max_iter=300, random_state=42),
}

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
scorer = make_scorer(f1_score, average="macro")

results_10fold = []
for name, model in models.items():
    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", model)
    ])
    if len(X) == 0:
        continue
    scores = cross_val_score(pipeline, X, y, cv=cv, scoring=scorer, n_jobs=-1)
    results_10fold.append((name, scores.mean(), scores.std()))

print("10-fold cross-validation (macro F1) :")
for name, mean_score, std_score in results_10fold:
    print(f"{name:15s} mean={mean_score:.3f} std={std_score:.3f}")

print("\nBalance des classes :")
print(y.value_counts().sort_index())

10-fold cross-validation (macro F1) :
LDA             mean=0.509 std=0.011
SVM             mean=0.650 std=0.012
RandomForest    mean=0.743 std=0.012
KNN             mean=0.699 std=0.013
NaiveBayes      mean=0.412 std=0.012
DecisionTree    mean=0.650 std=0.015
AdaBoost        mean=0.586 std=0.023
MLP             mean=0.718 std=0.013

Balance des classes :
Label_Binary
0    5304
1    7248
Name: count, dtype: int64


## 9. Évaluation par validation croisée et par sujet

Deux évaluations sont demandées :

### 10-fold cross-validation

Les segments sont répartis en 10 folds stratifiés. Cette évaluation est utile pour comparer les modèles, mais elle peut mélanger les sujets entre apprentissage et test.

### Leave-One-Subject-Out, LOSO

Un sujet est laissé de côté pour le test, tandis que le modèle est entraîné sur les autres sujets. Cette stratégie d’évaluation est plus réaliste, car elle permet de tester la capacité de généralisation du modèle sur un conducteur jamais vu auparavant. L’opération est ensuite répétée sur l’ensemble des sujets disponibles afin d’obtenir une évaluation plus robuste.

## Question

Pourquoi le LOSO est-il souvent plus difficile que le 10-fold classique ?

### Réponse 

...

## 10. Interprétation et discussion

Répondez aux questions suivantes dans le notebook :

1. Quel modèle obtient le meilleur F1-score en 10-fold ?
2. Quel modèle obtient le meilleur F1-score en LOSO ?
3. Les performances chutent-elles en LOSO ? Pourquoi ?
4. Les classes sont-elles équilibrées ?
5. Les résultats obtenus avec EEG seul vous semblent-ils suffisants pour une application réelle ?
6. Quelles limites voyez-vous à l'utilisation des labels subjectifs PAAS ?
7. Quelles améliorations proposeriez-vous ?

## 11. Mini-système d'adaptation

À partir de la prédiction du modèle, on peut simuler une décision d'adaptation.

Exemple :

| Prédiction | Décision |
|---|---|
| charge faible | interface normale |
| charge élevée | simplification de l'interface |
| charge élevée persistante | alerte conducteur |

## Question 

Pourquoi faut-il être prudent avant de déclencher une alerte sur une seule prédiction ?

### Réponse 

...

In [10]:
def decision_system(predictions, probabilities=None, persistence_window=3):
    """
    Transforme les prédictions en décision d'adaptation.

    predictions: vecteur de labels binaires (0 faible, 1 élevé) ou probabilités.
    probabilities: si fourni, utilisé pour calculer les labels binaires.
    persistence_window: nombre de fenêtres consécutives de charge élevée
        nécessaires pour déclencher une alerte conducteur.
    """
    preds = np.asarray(predictions)
    single_value = preds.ndim == 0

    if probabilities is not None:
        probs = np.asarray(probabilities)
        if probs.shape != preds.shape:
            raise ValueError("probabilities doit avoir la même forme que predictions")
        preds = (probs >= 0.5).astype(int)
    else:
        if np.issubdtype(preds.dtype, np.floating):
            preds = (preds >= 0.5).astype(int)
        else:
            preds = preds.astype(int)

    preds = np.atleast_1d(preds)
    decisions = []
    high_count = 0
    for pred in preds:
        if pred == 0:
            decisions.append("interface normale")
            high_count = 0
        else:
            high_count += 1
            if high_count >= persistence_window:
                decisions.append("alerte conducteur")
            else:
                decisions.append("simplification de l'interface")

    return decisions[0] if single_value else decisions


## 12. Extension optionnelle — vers la multimodalité

Le cœur du TP est volontairement limité à l'EEG.

Une extension possible consiste à reproduire les mêmes étapes pour les autres modalités :

```text
ECG_Features_10s → Normalized_Features_10s/ECG → Normalized_Features_10s_With_Label/ECG
EDA_Features_10s → Normalized_Features_10s/EDA → Normalized_Features_10s_With_Label/EDA
Gaze_Features_10s → Normalized_Features_10s/Gaze → Normalized_Features_10s_With_Label/Gaze
```

Puis à fusionner les features :

```text
EEG + ECG
EEG + EDA
EEG + Gaze
EEG + ECG + EDA + Gaze
```

La fusion la plus simple est une concaténation des colonnes de features pour des fenêtres correspondant au même sujet, au même niveau et au même indice de fenêtre.

## Question

Pourquoi la multimodalité peut-elle améliorer la détection de la charge cognitive ?

### Réponse 

...